# Knowledge Distillation
- The concept of **knowledge distillation** is to utilize class probabilities of a higher-capacity model (teacher) as soft targets of a smaller model (student)
- The implement processes can be divided into several stages:
  1. Finish the `ResNet()` classes
  2. Train the teacher model (ResNet50) and the student model (ResNet18) from scratch, i.e. **without KD**
  3. Define the `Distiller()` class and `loss_re()`, `loss_fe()` functions
  4. Train the student model **with KD** from the teacher model in two different ways, response-based and feature based distillation
  5. Comparison of student models w/ & w/o KD

## Setup

In [ ]:
! pip install torchinfo

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, random_split
from torchinfo import summary
from tqdm import tqdm
import sys
import numpy as np
import math
import matplotlib.pyplot as plt
import os
from PIL import Image

In [ ]:
torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

## Download dataset

In [ ]:
validation_split = 0.2
batch_size = 128

# data augmentation and normalization
transform_train = transforms.Compose([
                    transforms.RandomCrop(32, padding=4),
                    transforms.RandomHorizontalFlip(),
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

transform_test = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# download dataset
train_and_val_dataset = torchvision.datasets.CIFAR10(
    root='dataset/',
    train=True,
    transform=transform_train,
    download=True
)

test_dataset = torchvision.datasets.CIFAR10(
    root='dataset/',
    train=False,
    transform=transform_test,
    download=True
)

# split train and validation dataset
train_size = int((1 - validation_split) * len(train_and_val_dataset))
val_size = len(train_and_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_and_val_dataset, [train_size, val_size])

# create dataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

test_num = len(test_dataset)
test_steps = len(test_loader)

## Create teacher and student models
### Define BottleNeck for ResNet50

In [ ]:
class BottleNeck(nn.Module):
    expansion = 4

    def __init__(self, in_channel, out_channel, stride=1, downsample=None, **kwargs):
        super(BottleNeck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=out_channel, kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(in_channels=out_channel, out_channels=out_channel, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)
        self.conv3 = nn.Conv2d(in_channels=out_channel, out_channels=out_channel * self.expansion, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channel * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out += identity
        out = self.relu(out)

        return out

### Define Resifual Block

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channel, out_channel, stride=1, downsample=None, **kwargs):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=out_channel, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=out_channel, out_channels=out_channel, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

### Define ResNet Model

In [ ]:
class ResNet(nn.Module):

    def __init__(self, block, blocks_num, num_classes=1000):
        super(ResNet, self).__init__()
        self.in_channel = 64

        self.conv1 = nn.Conv2d(3, self.in_channel, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_channel)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, blocks_num[0])
        self.layer2 = self._make_layer(block, 128, blocks_num[1], stride=2)
        self.layer3 = self._make_layer(block, 256, blocks_num[2], stride=2)
        self.layer4 = self._make_layer(block, 512, blocks_num[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def _make_layer(self, block, channel, block_num, stride=1):
        downsample = None
        if stride != 1 or self.in_channel != channel * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channel, channel * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(channel * block.expansion))

        layers = []
        layers.append(block(self.in_channel, channel, downsample=downsample, stride=stride))
        self.in_channel = channel * block.expansion

        for _ in range(1, block_num):
            layers.append(block(self.in_channel, channel))

        return nn.Sequential(*layers)

    def forward(self, x):
        # 1. Finish the forward pass and return the output layer as well as hidden features.
        # 2. The output layer and hidden features will be used later for distilling.
        # 3. You can refer to the ResNet structure illustration to finish it.
        # Initial convolution and max pooling
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # ResNet Layers - capturing intermediate features
        feature1 = self.layer1(x)
        feature2 = self.layer2(feature1)
        feature3 = self.layer3(feature2)
        feature4 = self.layer4(feature3)

        # Final classification head
        x = self.avgpool(feature4)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        # Return logits and the list of features as required by the notebook
        return x, [feature1, feature2, feature3, feature4]

### Define ResNet50 and Resnet18

In [ ]:
def resnet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

def resnet50(num_classes=10):
    return ResNet(BottleNeck, [3, 4, 6, 3], num_classes=num_classes)

## Teacher Model (ResNet50)

In [ ]:
Teacher = resnet50(num_classes=10)  # commment out this line if loading trained teacher model
# Teacher = torch.load('Teacher.pt', weights_only=False)  # loading trained teacher model
Teacher = Teacher.to(device)

In [ ]:
summary(Teacher)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            1,728
├─BatchNorm2d: 1-2                       128
├─ReLU: 1-3                              --
├─MaxPool2d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BottleNeck: 2-1                   --
│    │    └─Conv2d: 3-1                  4,096
│    │    └─BatchNorm2d: 3-2             128
│    │    └─Conv2d: 3-3                  36,864
│    │    └─BatchNorm2d: 3-4             128
│    │    └─Conv2d: 3-5                  16,384
│    │    └─BatchNorm2d: 3-6             512
│    │    └─ReLU: 3-7                    --
│    │    └─Sequential: 3-8              16,896
│    └─BottleNeck: 2-2                   --
│    │    └─Conv2d: 3-9                  16,384
│    │    └─BatchNorm2d: 3-10            128
│    │    └─Conv2d: 3-11                 36,864
│    │    └─BatchNorm2d: 3-12            128
│    │    └─Conv2d: 3-13               

## Student Model (ResNet18)

In [ ]:
Student = resnet18(num_classes=10)  # commment out this line if loading trained student model
# Student = torch.load('Student.pt', weights_only=False)  # loading trained student model
Student = Student.to(device)

In [ ]:
summary(Student)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            1,728
├─BatchNorm2d: 1-2                       128
├─ReLU: 1-3                              --
├─MaxPool2d: 1-4                         --
├─Sequential: 1-5                        --
│    └─BasicBlock: 2-1                   --
│    │    └─Conv2d: 3-1                  36,864
│    │    └─BatchNorm2d: 3-2             128
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  36,864
│    │    └─BatchNorm2d: 3-5             128
│    └─BasicBlock: 2-2                   --
│    │    └─Conv2d: 3-6                  36,864
│    │    └─BatchNorm2d: 3-7             128
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  36,864
│    │    └─BatchNorm2d: 3-10            128
├─Sequential: 1-6                        --
│    └─BasicBlock: 2-3                   --
│    │    └─Conv2d: 3-11                 73,728

## Define training function

In [ ]:
def train_from_scratch(model, train_loader, val_loader, epochs, learning_rate, device, model_name):
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=learning_rate)

    loss = []
    train_error=[]
    val_error = []
    valdation_error = []
    train_loss = []
    valdation_loss = []
    train_accuraacy = []
    valdation_accuracy= []

    for epoch in range(epochs):
        train_loss = 0.0
        valid_loss = 0.0
        train_acc = 0.0
        valid_acc = 0.0
        correct = 0.
        total = 0.
        V_correct = 0.
        V_total = 0.

        model.train()
        train_bar = tqdm(train_loader, file=sys.stdout)
        for step, data in enumerate(train_bar):
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits, hidden = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
            pred = logits.data.max(1, keepdim=True)[1]
            correct += np.sum(np.squeeze(pred.eq(labels.data.view_as(pred))).cpu().numpy())
            total += images.size(0)
            train_acc =  correct/total
            train_bar.desc = "train epoch[{}/{}]".format(epoch + 1, epochs)

        model.eval()
        with torch.no_grad():
            val_bar = tqdm(val_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_labels = val_data
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                outputs, hidden_outputs = model(val_images)
                loss = criterion(outputs, val_labels)
                valid_loss += loss.item() * val_images.size(0)
                pred = outputs.data.max(1, keepdim=True)[1]
                V_correct += np.sum(np.squeeze(pred.eq(val_labels.data.view_as(pred))).cpu().numpy())
                V_total += val_images.size(0)
                val_bar.desc = "valid epoch[{}/{}]".format(epoch + 1, epochs)

        train_loss = train_loss / len(train_loader.dataset)
        train_error.append(train_loss)
        valid_loss = valid_loss / len(val_loader.dataset)
        val_error.append(valid_loss)
        train_accuraacy.append( correct / total)
        valdation_accuracy.append(V_correct / V_total)

        print('\tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(train_loss, valid_loss))
        print('\tTrain Accuracy: %.3fd%% (%2d/%2d)\tValdation Accuracy: %.3fd%% (%2d/%2d) '% (100. * correct / total, correct, total, 100. * V_correct / V_total, V_correct, V_total))

    torch.save(model, f'{model_name}.pt')
    print(f'{model_name}.pt is saved')

    print('Finished Training')

## Define testing function

In [ ]:
def test(model, test_loader ,device, type=None):
    criterion = nn.CrossEntropyLoss()
    acc = 0.0
    test_loss = 0.0

    if type == None:
        model.eval()
    elif type == 'distiller':
        model.eval()
        model.teacher.eval()
        model.student.eval()
    else:
       raise ValueError(f'Error: only support response-based and feature-based distillation')

    with torch.no_grad():
        test_bar = tqdm(test_loader, file=sys.stdout)
        for test_data in test_bar:
            test_images, test_labels = test_data
            test_images, test_labels = test_images.to(device), test_labels.to(device)
            if type == None:
                outputs, features = model(test_images)
                loss = criterion(outputs, test_labels)
            elif type == 'distiller':
                outputs, loss = model(test_images, test_labels)
            else:
                raise ValueError(f'Error: only support response-based and feature-based distillation')

            predict_y = torch.max(outputs, dim=1)[1]
            acc += torch.eq(predict_y, test_labels.to(device)).sum().item()
            test_loss += loss.item()
            test_bar.desc = "test"

    test_accurate = acc / test_num
    print('test_loss: %.3f  test_accuracy: %.3f' %(test_loss / test_steps, test_accurate * 100))
    return test_loss / test_steps, test_accurate * 100.

## Train Teacher and Student model from scratch

In [ ]:
# Decide the epochs and learning rate
train_from_scratch(Teacher, train_loader, val_loader, epochs=30 , learning_rate=1e-3 , device=device, model_name="Teacher")

valid epoch[1/30]: 100%|██████████| 79/79 [00:05<00:00, 13.73it/s]
	Training Loss: 1.692187 	Validation Loss: 1.392161
	Train Accuracy: 39.095d% (15638/40000)	Valdation Accuracy: 49.950d% (4995/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:06<00:00, 12.66it/s]
	Training Loss: 1.251190 	Validation Loss: 1.191249
	Train Accuracy: 55.585d% (22234/40000)	Valdation Accuracy: 58.470d% (5847/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:05<00:00, 13.32it/s]
	Training Loss: 1.026997 	Validation Loss: 1.128924
	Train Accuracy: 63.888d% (25555/40000)	Valdation Accuracy: 61.810d% (6181/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:05<00:00, 13.49it/s]
	Training Loss: 0.869367 	Validation Loss: 0.942905
	Train Accuracy: 69.728d% (27891/40000)	Valdation Accuracy: 67.680d% (6768/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:06<00:00, 12.62it/s]
	Training Loss: 0.771423 	Validation Loss: 0.841598
	Train Accuracy: 73.052d% (29221/40000)	Valdation Accuracy: 70.950d% (709

In [ ]:
T_loss, T_accuracy = test(Teacher, test_loader, device=device)

test: 100%|██████████| 79/79 [00:04<00:00, 17.56it/s]
test_loss: 0.427  test_accuracy: 86.600


In [ ]:
# Decide the epochs and learning rate
train_from_scratch(Student, train_loader, val_loader, epochs=30 , learning_rate=1e-3 , device=device, model_name="Student")

valid epoch[1/30]: 100%|██████████| 79/79 [00:04<00:00, 16.61it/s]
	Training Loss: 1.401694 	Validation Loss: 1.302076
	Train Accuracy: 48.905d% (19562/40000)	Valdation Accuracy: 55.720d% (5572/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:04<00:00, 17.34it/s]
	Training Loss: 1.010594 	Validation Loss: 1.003114
	Train Accuracy: 64.270d% (25708/40000)	Valdation Accuracy: 65.910d% (6591/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:05<00:00, 15.44it/s]
	Training Loss: 0.841173 	Validation Loss: 0.862946
	Train Accuracy: 70.547d% (28219/40000)	Valdation Accuracy: 69.800d% (6980/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:04<00:00, 17.60it/s]
	Training Loss: 0.729903 	Validation Loss: 0.909229
	Train Accuracy: 74.457d% (29783/40000)	Valdation Accuracy: 69.260d% (6926/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:05<00:00, 15.54it/s]
	Training Loss: 0.641313 	Validation Loss: 0.877195
	Train Accuracy: 77.875d% (31150/40000)	Valdation Accuracy: 71.070d% (710

In [ ]:
S_loss, S_accuracy = test(Student, test_loader, device=device)

test: 100%|██████████| 79/79 [00:03<00:00, 20.69it/s]
test_loss: 0.449  test_accuracy: 87.320


## Define distillation

### Define the loss functions

In [ ]:
# Finish the loss function for response-based distillation.
def loss_re(student_logits, teacher_logits, labels):
    T = 4.0 # Temperature
    alpha = 0.5 # Weighting parameter

    # Distillation Loss (KL Divergence)
    ce = F.cross_entropy(student_logits, labels)

    # KL divergence between softened teacher and student predictions
    # Note: multiply by T^2 as in Hinton et al., to compensate for gradients
    log_p_student = F.log_softmax(student_logits / T, dim=1)
    p_teacher = F.softmax(teacher_logits / T, dim=1)
    kd = F.kl_div(log_p_student, p_teacher, reduction="batchmean") * (T * T)

    loss = alpha * kd + (1.0 - alpha) * ce
    return loss

In [ ]:
# Finish the loss function for feature-based distillation.
def loss_fe(student_logits, student_features, teacher_features, labels, alpha_f: float = 1e-3):
# Standard cross-entropy with ground-truth labels
    ce = F.cross_entropy(student_logits, labels)

    # Match intermediate features between teacher and student.
    # We use mean squared error (L2) between feature maps.
    feat_loss = 0.0
    for s_feat, t_feat in zip(student_features, teacher_features):
        # Align spatial dimensions if they differ slightly (e.g., due to pooling)
        if s_feat.shape != t_feat.shape:
            # Use adaptive pooling on teacher features to match student features
            t_feat_resized = F.adaptive_avg_pool2d(t_feat, s_feat.shape[-2:])
        else:
            t_feat_resized = t_feat
        feat_loss = feat_loss + F.mse_loss(s_feat, t_feat_resized)

    loss = ce + alpha_f * feat_loss
    return loss

### Define Distillation Framework

In [ ]:
class Distiller(nn.Module):
    def __init__(self, teacher, student, type):
        super(Distiller, self).__init__()

        # 1. Finish the __init__ method.
        # Teacher and student networks
        self.teacher = teacher
        self.student = student
        # Distillation type: "response" or "feature"
        self.type = type

        # Freeze teacher parameters; we do not update teacher during distillation
        for p in self.teacher.parameters():
            p.requires_grad = False

        # Add feature projectors for feature-based distillation
        if self.type == 'feature':
            student_channels = [64, 128, 256, 512]
            teacher_channels = [256, 512, 1024, 2048]
            self.feature_projectors = nn.ModuleList()
            for s_ch, t_ch in zip(student_channels, teacher_channels):
                self.feature_projectors.append(nn.Conv2d(s_ch, t_ch, kernel_size=1))

            # Move projectors to the device
            for proj in self.feature_projectors:
                proj.to(device)

    def forward(self, x, target):
        # 2. Finish the forward pass.
        # Get Teacher outputs
        with torch.no_grad():
            teacher_logits, teacher_features = self.teacher(x)

        # Student predictions
        student_logits, student_features = self.student(x)

        if self.type == 'response':
            loss_distill = loss_re(student_logits, teacher_logits, target) # call the loss_re()
        elif self.type == 'feature':
            # Apply feature projectors to student features
            projected_student_features = [proj(s_feat) for proj, s_feat in zip(self.feature_projectors, student_features)]
            loss_distill = loss_fe(student_logits, projected_student_features, teacher_features, target) # call the loss_fe()
        else:
            raise ValueError(f'Error: only support response-based and feature-based distillation')

        return student_logits, loss_distill

### Training function

In [ ]:
def train_distillation(distiller, student, train_loader, val_loader, epochs, learning_rate, device):
    ce_loss = nn.CrossEntropyLoss()
    # define the parameter the optimizer used
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, distiller.parameters()), lr=learning_rate)

    loss = []
    train_error=[]
    val_error = []
    valdation_error = []
    train_loss = []
    valdation_loss = []
    train_accuraacy = []
    valdation_accuracy= []

    for epoch in range(epochs):
        distiller.train()
        distiller.teacher.train()
        distiller.student.train()

        train_loss = 0.0
        valid_loss = 0.0
        train_acc = 0.0
        valid_acc  = 0.0
        correct = 0.
        total = 0.
        V_correct = 0.
        V_total = 0.
        train_bar = tqdm(train_loader, file=sys.stdout)
        for step, data in enumerate(train_bar):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            outputs, loss = distiller(images, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            pred = outputs.data.max(1, keepdim=True)[1]
            result = pred.eq(labels.data.view_as(pred))
            result = np.squeeze(result.cpu().numpy())
            correct += np.sum(result)
            total += images.size(0)
            train_bar.desc = "train epoch[{}/{}]".format(epoch + 1, epochs)

        distiller.eval()
        distiller.teacher.eval()
        distiller.student.eval()

        with torch.no_grad():
            val_bar = tqdm(val_loader, file=sys.stdout)
            for val_data in val_bar:

                val_images, val_labels = val_data
                val_images, val_labels = val_images.to(device), val_labels.to(device)

                outputs, loss = distiller(val_images, val_labels)

                valid_loss += loss.item() * val_images.size(0)
                pred = outputs.max(1, keepdim=True)[1]
                V_correct += np.sum(np.squeeze(pred.eq(val_labels.data.view_as(pred))).cpu().numpy())
                V_total += val_images.size(0)
                val_bar.desc = "valid epoch[{}/{}]".format(epoch + 1, epochs)

        train_loss = train_loss / len(train_loader.dataset)
        train_error.append(train_loss)
        valid_loss = valid_loss / len(val_loader.dataset)
        val_error.append(valid_loss)
        train_accuraacy.append( correct / total)
        valdation_accuracy.append(V_correct / V_total)

        print('\tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(train_loss, valid_loss))
        print('\tTrain Accuracy: %.3fd%% (%2d/%2d)\tValdation Accuracy: %.3fd%% (%2d/%2d) '% (100. * correct / total, correct, total, 100. * V_correct / V_total, V_correct, V_total))

    print('Finished Distilling')

## Response-based distillation

In [ ]:
# Decide the epochs and learning rate
Student_re = resnet18(num_classes=10)
Student_re = Student_re.to(device)
distiller_re = Distiller(Teacher, Student_re, type='response')
train_distillation(distiller_re, Student_re, train_loader, val_loader, epochs=30 , learning_rate=1e-3 , device=device)

valid epoch[1/30]: 100%|██████████| 79/79 [00:07<00:00, 10.93it/s]
	Training Loss: 4.130385 	Validation Loss: 3.326224
	Train Accuracy: 50.425d% (20170/40000)	Valdation Accuracy: 55.720d% (5572/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:06<00:00, 11.77it/s]
	Training Loss: 2.318855 	Validation Loss: 2.246613
	Train Accuracy: 67.162d% (26865/40000)	Valdation Accuracy: 67.070d% (6707/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:06<00:00, 11.42it/s]
	Training Loss: 1.635588 	Validation Loss: 1.992095
	Train Accuracy: 74.270d% (29708/40000)	Valdation Accuracy: 69.460d% (6946/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:06<00:00, 11.94it/s]
	Training Loss: 1.292181 	Validation Loss: 1.417687
	Train Accuracy: 78.460d% (31384/40000)	Valdation Accuracy: 76.320d% (7632/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:06<00:00, 11.54it/s]
	Training Loss: 1.081662 	Validation Loss: 1.168432
	Train Accuracy: 80.605d% (32242/40000)	Valdation Accuracy: 78.660d% (786

In [ ]:
reS_loss, reS_accuracy = test(distiller_re, test_loader, type='distiller', device=device)

test: 100%|██████████| 79/79 [00:05<00:00, 15.03it/s]
test_loss: 0.571  test_accuracy: 87.270


## Feature-based distillation

In [70]:
# Decide the epochs and learning rate
Student_fe = resnet18(num_classes=10)
Student_fe = Student_fe.to(device)
distiller_fe = Distiller(Teacher, Student_fe, type='feature')
train_distillation(distiller_fe, Student_fe, train_loader, val_loader, epochs=30 , learning_rate=1e-3 , device=device)

valid epoch[1/30]: 100%|██████████| 79/79 [00:06<00:00, 11.49it/s]
	Training Loss: 1.408725 	Validation Loss: 1.199844
	Train Accuracy: 48.740d% (19496/40000)	Valdation Accuracy: 57.660d% (5766/10000) 
valid epoch[2/30]: 100%|██████████| 79/79 [00:07<00:00, 11.02it/s]
	Training Loss: 0.996027 	Validation Loss: 0.944407
	Train Accuracy: 65.203d% (26081/40000)	Valdation Accuracy: 67.630d% (6763/10000) 
valid epoch[3/30]: 100%|██████████| 79/79 [00:07<00:00, 11.12it/s]
	Training Loss: 0.814742 	Validation Loss: 0.909305
	Train Accuracy: 71.690d% (28676/40000)	Valdation Accuracy: 68.580d% (6858/10000) 
valid epoch[4/30]: 100%|██████████| 79/79 [00:06<00:00, 11.82it/s]
	Training Loss: 0.715506 	Validation Loss: 0.763500
	Train Accuracy: 75.573d% (30229/40000)	Valdation Accuracy: 73.980d% (7398/10000) 
valid epoch[5/30]: 100%|██████████| 79/79 [00:06<00:00, 11.56it/s]
	Training Loss: 0.640853 	Validation Loss: 0.726553
	Train Accuracy: 78.183d% (31273/40000)	Valdation Accuracy: 75.640d% (756

In [71]:
ftS_loss, ftS_accuracy = test(distiller_fe, test_loader, type='distiller', device=device)

test: 100%|██████████| 79/79 [00:05<00:00, 14.40it/s]
test_loss: 0.533  test_accuracy: 85.890


## Result and Comparison

In [72]:
print(f'Teacher from scratch: loss = {T_loss:.2f}, accuracy = {T_accuracy:.2f}')
print(f'Student from scratch: loss = {S_loss:.2f}, accuracy = {S_accuracy:.2f}')
print(f'Response-based student: loss = {reS_loss:.2f}, accuracy = {reS_accuracy:.2f}')
print(f'Featured-based student: loss = {ftS_loss:.2f}, accuracy = {ftS_accuracy:.2f}')

Teacher from scratch: loss = 0.43, accuracy = 86.60
Student from scratch: loss = 0.45, accuracy = 87.32
Response-based student: loss = 0.57, accuracy = 87.27
Featured-based student: loss = 0.53, accuracy = 85.89
